In [1]:
import sys
print(sys.executable)


/Library/Developer/CommandLineTools/usr/bin/python3


In [2]:
import graphviz
print("graphviz found!")

graphviz found!


In [19]:
from graphviz import Digraph
from IPython.display import display, HTML
import json

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, rankdir='LR'):
    nodes, edges = trace(root)
    dot = Digraph(graph_attr={'rankdir': rankdir})
    for n in nodes:
        uid = str(id(n))
       # dot.node(name=uid, label=f"{{ {n.label} | data {n.data:.4f} | grad {n.grad:.4f} }}", shape='record')
        dot.node(name = uid, label = "{ data %.4f }" % (n.data, ), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot.source  # return DOT source text, not a rendered object

def render_dot_wasm(dot_source, height=400):
    html = f"""
    <div id="graph" style="text-align:center;"></div>
    <script src="https://cdn.jsdelivr.net/npm/@hpcc-js/wasm/dist/index.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
    <script src="https://cdn.jsdelivr.net/npm/d3-graphviz@5"></script>
    <script>
      d3.select("#graph").graphviz()
        .renderDot(`{dot_source}`);
    </script>
    """
    display(HTML(html))

# usage
#dot_source = draw_dot(your_root_value_node)
#render_dot_wasm(dot_source)

In [7]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

Matplotlib is building the font cache; this may take a moment.


In [2]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
    
    def __repr__(self):
        return f"Value(data={self.data})"  
    
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out


a = Value(2.0)
b = Value(-3.0)
c = Value (10.0)

d = a * b + c
d
#(a.__mul__(b)).__add__(c)


Value(data=4.0)

In [1]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges


def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    # dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    dot.node(name = uid, label = "{ data %.4f }" % (n.data, ), shape='record')  
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot
#draw_dot(d)

In [20]:
dot_source = draw_dot(d)
render_dot_wasm(dot_source)

In [21]:
print(render_dot_wasm)

<function render_dot_wasm at 0x10e37e430>


In [22]:
dot_source = draw_dot(d)
print(dot_source)

digraph {
	graph [rankdir=LR]
	4533644832 [label="{ data 4.0000 }" shape=record]
	"4533644832+" [label="+"]
	"4533644832+" -> 4533644832
	4533644928 [label="{ data -6.0000 }" shape=record]
	"4533644928*" [label="*"]
	"4533644928*" -> 4533644928
	4533644976 [label="{ data -3.0000 }" shape=record]
	4533645024 [label="{ data 10.0000 }" shape=record]
	4533645072 [label="{ data 2.0000 }" shape=record]
	4533644976 -> "4533644928*"
	4533645024 -> "4533644832+"
	4533645072 -> "4533644928*"
	4533644928 -> "4533644832+"
}



In [4]:
from graphviz import Digraph
draw_dot(d)  # renders inline automatically in Jupyter

NameError: name 'draw_dot' is not defined

In [7]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})
    for n in nodes:
        uid = str(id(n))
        #dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        dot.node(name = uid, label = "{ data %.4f }" % (n.data, ), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

In [8]:
draw_dot(d)  # renders inline automatically in Jupyter


ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH